In [2]:
!pip install -q transformers datasets accelerate

our dataset might contain explanations of Linux, Docker, AWS, and networking.


In [3]:
# Step 3: Create a custom text dataset

training_text = """
Linux is an operating system widely used for servers.
It provides tools for managing files, users, processes, and services.

Networking allows computers and other devices to communicate.
An IP address identifies a network interface on an IP network.
DNS translates domain names into IP addresses.

Docker is a containerization platform.
It packages applications and their dependencies into containers.
Containers help applications run consistently across different environments.

Kubernetes is a container orchestration platform.
It helps deploy, scale, and manage containerized applications.

DevOps combines development and operations practices.
It helps teams automate software development, testing, deployment, and monitoring.

AWS is a cloud computing platform.
It provides services such as virtual servers, storage, networking, and databases.

An HTTP request allows a client to communicate with a web server.
HTTPS uses TLS to protect HTTP communication in transit.

A load balancer distributes incoming requests across multiple servers.
This can improve application availability and distribute traffic.

An auto scaling group can automatically adjust the number of virtual machines
based on configured scaling policies.

SSH is a protocol used to securely access remote computers.
It is commonly used by administrators to manage Linux servers.
"""

# Save the dataset to a text file
with open("training_data.txt", "w", encoding="utf-8") as file:
    file.write(training_text.strip())

print("Dataset created successfully!")
print("Number of characters:", len(training_text))
print("\nDataset preview:\n")
print(training_text[:500])

Dataset created successfully!
Number of characters: 1353

Dataset preview:


Linux is an operating system widely used for servers.
It provides tools for managing files, users, processes, and services.

Networking allows computers and other devices to communicate.
An IP address identifies a network interface on an IP network.
DNS translates domain names into IP addresses.

Docker is a containerization platform.
It packages applications and their dependencies into containers.
Containers help applications run consistently across different environments.

Kubernetes is a con


Verify that the file exists

In [4]:
import os

print("File exists:", os.path.exists("training_data.txt"))

with open("training_data.txt", "r", encoding="utf-8") as file:
    print("\nDataset contents:\n")
    print(file.read())

File exists: True

Dataset contents:

Linux is an operating system widely used for servers.
It provides tools for managing files, users, processes, and services.

Networking allows computers and other devices to communicate.
An IP address identifies a network interface on an IP network.
DNS translates domain names into IP addresses.

Docker is a containerization platform.
It packages applications and their dependencies into containers.
Containers help applications run consistently across different environments.

Kubernetes is a container orchestration platform.
It helps deploy, scale, and manage containerized applications.

DevOps combines development and operations practices.
It helps teams automate software development, testing, deployment, and monitoring.

AWS is a cloud computing platform.
It provides services such as virtual servers, storage, networking, and databases.

An HTTP request allows a client to communicate with a web server.
HTTPS uses TLS to protect HTTP communication i

Load the required libraries

In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM

print("Libraries imported successfully!")

Libraries imported successfully!


Load the GPT-2 tokenizer

In [6]:
# Load the GPT-2 tokenizer

model_name = "openai-community/gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)

print("GPT-2 tokenizer loaded successfully!")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

GPT-2 tokenizer loaded successfully!


Test the tokenizer

In [7]:
text = "Docker is useful for application deployment."

# Convert text into token IDs
tokens = tokenizer.encode(text)

print("Original text:", text)
print("Token IDs:", tokens)
print("Number of tokens:", len(tokens))

Original text: Docker is useful for application deployment.
Token IDs: [35, 12721, 318, 4465, 329, 3586, 14833, 13]
Number of tokens: 8


Load the GPT-2 model


In [8]:
# Load the pretrained GPT-2 model

model = AutoModelForCausalLM.from_pretrained(model_name)

print("GPT-2 model loaded successfully!")

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

GPT-2 model loaded successfully!


Test GPT-2 before fine-tuning

In [9]:
import torch

prompt = "Linux is"

inputs = tokenizer(prompt, return_tensors="pt")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=40,
        do_sample=True,
        temperature=0.8,
        top_k=50,
        pad_token_id=tokenizer.eos_token_id
    )

generated_text = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("Prompt:", prompt)
print("\nGenerated text:\n")
print(generated_text)

Prompt: Linux is

Generated text:

Linux is getting better, and it's more than capable of delivering on this promise.

A new feature in Android 6.0 Lollipop called "Gingerbread" was released in May, but even


In [10]:
# Step 5.2: Read the training dataset

with open("training_data.txt", "r", encoding="utf-8") as file:
    training_text = file.read()

print("Dataset loaded successfully!")
print("Total characters:", len(training_text))
print("\nDataset preview:\n")
print(training_text[:300])

Dataset loaded successfully!
Total characters: 1351

Dataset preview:

Linux is an operating system widely used for servers.
It provides tools for managing files, users, processes, and services.

Networking allows computers and other devices to communicate.
An IP address identifies a network interface on an IP network.
DNS translates domain names into IP addresses.

Do


Tokenize the dataset

In [11]:
# Step 5.3: Convert text into token IDs

token_ids = tokenizer.encode(training_text)

print("Tokenization completed!")
print("Total tokens:", len(token_ids))
print("\nFirst 30 token IDs:")
print(token_ids[:30])

Tokenization completed!
Total tokens: 266

First 30 token IDs:
[19314, 318, 281, 5361, 1080, 6768, 973, 329, 9597, 13, 198, 1026, 3769, 4899, 329, 11149, 3696, 11, 2985, 11, 7767, 11, 290, 2594, 13, 198, 198, 7934, 16090, 3578]


Let's convert the token IDs back into readable text.

In [12]:
# Step 5.4: Decode the token IDs

decoded_text = tokenizer.decode(token_ids)

print("Original text:")
print(training_text[:200])

print("\nDecoded text:")
print(decoded_text[:200])

Original text:
Linux is an operating system widely used for servers.
It provides tools for managing files, users, processes, and services.

Networking allows computers and other devices to communicate.
An IP address

Decoded text:
Linux is an operating system widely used for servers.
It provides tools for managing files, users, processes, and services.

Networking allows computers and other devices to communicate.
An IP address


Before training, GPT-2 needs to know how to handle batches of token IDs.

We will configure its padding token.

In [13]:
# Step 5.5: Configure the tokenizer for training

tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id

print("Padding token configured!")
print("EOS token:", tokenizer.eos_token)
print("Padding token:", tokenizer.pad_token)
print("Padding token ID:", tokenizer.pad_token_id)

Padding token configured!
EOS token: <|endoftext|>
Padding token: <|endoftext|>
Padding token ID: 50256


Prepare the training dataset

In [14]:
from datasets import Dataset

# Convert our training text into token IDs
all_token_ids = tokenizer.encode(
    training_text,
    add_special_tokens=False
)

# Add the end-of-text token
all_token_ids.append(tokenizer.eos_token_id)

# Divide tokens into small sequences
block_size = 64
examples = []

for i in range(0, len(all_token_ids), block_size):
    chunk = all_token_ids[i:i + block_size]

    # Keep sequences containing at least 8 tokens
    if len(chunk) >= 8:
        examples.append({"input_ids": chunk})

# Convert sequences into a Hugging Face Dataset
train_dataset = Dataset.from_list(examples)

print("Training dataset prepared successfully!")
print("Total tokens:", len(all_token_ids))
print("Training examples:", len(train_dataset))
print("\nFirst training example:")
print(tokenizer.decode(train_dataset[0]["input_ids"]))

Training dataset prepared successfully!
Total tokens: 267
Training examples: 5

First training example:
Linux is an operating system widely used for servers.
It provides tools for managing files, users, processes, and services.

Networking allows computers and other devices to communicate.
An IP address identifies a network interface on an IP network.
DNS translates domain names into IP addresses.

Docker


Prepare the training data collator

In [15]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

print("Data collator prepared successfully!")

Data collator prepared successfully!


Configure the training

In [16]:
import torch
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./gpt2-devops-finetuned",
    num_train_epochs=5,
    per_device_train_batch_size=2,
    learning_rate=5e-5,
    weight_decay=0.01,
    logging_steps=1,
    save_strategy="no",
    report_to="none",
    fp16=torch.cuda.is_available()
)

print("Training configuration created successfully!")
print("GPU available:", torch.cuda.is_available())

Training configuration created successfully!
GPU available: True


start training

In [17]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator
)

print("Starting GPT-2 fine-tuning...")

trainer.train()

print("Fine-tuning completed!")

Starting GPT-2 fine-tuning...


[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
1,3.536127
2,3.931596
3,3.256106
4,4.045270
5,3.772401
6,3.094614
7,2.818530
8,3.620040
9,2.627075
10,2.952124


Fine-tuning completed!


Save the model and tokenizer

In [22]:
save_directory = "./gpt2-devops-finetuned"

# Save the fine-tuned model
model.save_pretrained(save_directory)

# Save the tokenizer
tokenizer.save_pretrained(save_directory)

print("Model and tokenizer saved successfully!")
print("Saved location:", save_directory)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model and tokenizer saved successfully!
Saved location: ./gpt2-devops-finetuned


Download your model to your computer

In [23]:
import shutil
from google.colab import files

shutil.make_archive(
    "/content/gpt2-devops-finetuned",
    "zip",
    "/content/gpt2-devops-finetuned"
)

files.download("/content/gpt2-devops-finetuned.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Run the generation code

In [24]:
import torch

# Put the trained model into evaluation mode
model.eval()

# Give the model a prompt
prompt = "AWS provides"

# Convert the prompt into tokens
inputs = tokenizer(prompt, return_tensors="pt")

# Move input tokens to the model's device
inputs = inputs.to(model.device)

# Generate text
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=True,
        temperature=0.7,
        top_k=40,
        pad_token_id=tokenizer.eos_token_id
    )

# Convert generated tokens back into text
generated_text = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("Prompt:", prompt)
print("\nGenerated text:\n")
print(generated_text)

Prompt: AWS provides

Generated text:

AWS provides a secure way to encrypt communication between users and servers.

The application is built with the following dependencies:

Python 3.7

Cython

Git

Java

PHP

Python 2.7

